In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# 02 — Preprocessing

Prepares the standardized inference modeling dataset created by `01_EDA.ipynb` for machine learning:

1. Load the execution level inference dataset created during EDA.
2. Document the feature inventory and run an explicit leakage audit.
3. Perform a stratified 70/15/15 train/validation/test split **before** any preprocessing is fitted.
4. Fit preprocessing (median imputation + missing indicators + scaling; one hot encoding) on **training data only**.
5. Validate the processed datasets to ensure there are no non finite values or partition overlap.
6. Persist processed partitions to `/content/processed/` for `03_Modeling.ipynb`.

## 2.1 Feature Inventory

The modeling dataset contains workload and configuration variables that can be known before or at the start of an execution.
The feature inventory below documents the role of each field and identifies variables that are excluded because they are targets, execution outcomes, identifiers, or potential sources of data leakage.
The final predictor set includes:
- `workload_type`
- `batch_size`
- `max_output_tokens`
- `request_rate`
- `output_length`
- `dataset`

Workload specific fields are represented as missing values when they are not applicable to a particular workload type. These missing value patterns are retained during preprocessing because they provide information about the workload configuration.

In [2]:
import pandas as pd
feature_inventory = pd.DataFrame([
    {"field": "workload_type", "role": "Predictor", "reason": "Distinguishes workload behaviors."},
    {"field": "batch_size", "role": "Predictor", "reason": "Offline inference only; known before execution."},
    {"field": "max_output_tokens", "role": "Predictor", "reason": "Offline inference only; known before execution."},
    {"field": "request_rate", "role": "Predictor", "reason": "Online inference only; configured rate."},
    {"field": "output_length", "role": "Predictor", "reason": "Online-finite only; configured output length."},
    {"field": "dataset", "role": "Predictor", "reason": "Input dataset configuration (online)."},
    {"field": "num_prompts", "role": "Excluded", "reason": "Constant for online-finite; perfectly collinear with request_rate (corr=1.0) for online-rate."},
    {"field": "mean_power[W]", "role": "Target", "reason": "Execution-level mean power."},
    {"field": "peak_power[W]", "role": "Excluded", "reason": "Same-execution power outcome → target leakage."},
    {"field": "elapsed / execution_time_seconds", "role": "Excluded", "reason": "Measured during/after execution."},
    {"field": "throughput / latency metrics", "role": "Excluded", "reason": "Post-execution performance outcomes."},
    {"field": "completed / run_completed", "role": "Filtering only", "reason": "Used to drop incomplete runs, never a predictor."},
    {"field": "id / seed / repeat / path / timestamps / slurmid", "role": "Excluded", "reason": "Identifiers, not workload characteristics."},
])
display(feature_inventory)

,field,role,reason
0,workload_type,Predictor,Distinguishes workload behaviors.
1,batch_size,Predictor,Offline inference only; known before execution.
2,max_output_tokens,Predictor,Offline inference only; known before execution.
3,request_rate,Predictor,Online inference only; configured rate.
4,output_length,Predictor,Online-finite only; configured output length.
5,dataset,Predictor,Input dataset configuration (online).
6,num_prompts,Excluded,Constant for online-finite; perfectly collinea...
7,mean_power[W],Target,Execution-level mean power.
8,peak_power[W],Excluded,Same-execution power outcome → target leakage.
9,elapsed / execution_time_seconds,Excluded,Measured during/after execution.


## 2.2 Load the Standardized Inference Dataset

The execution level inference modeling dataset was created and standardized during `01_EDA.ipynb`.
This notebook loads that prepared dataset rather than rebuilding it from the raw metadata.

The dataset is stored at:

`/content/drive/MyDrive/data/processed/inference_modeling_dataset.csv`

The dataset contains the three inference workload types in a common schema, with `np.nan` used for workload specific fields that are not applicable.

In [3]:
from pathlib import Path
import pandas as pd
# Load the execution level modeling dataset created by 01_EDA.ipynb.
modeling_dataset_path = Path(
    "/content/drive/MyDrive/data/processed/inference_modeling_dataset.csv"
)
if not modeling_dataset_path.exists():
    raise FileNotFoundError(
        f"Modeling dataset not found at: {modeling_dataset_path}"
    )
inference_dataset = pd.read_csv(modeling_dataset_path)
print(f"Dataset shape: {inference_dataset.shape}")
print("\nWorkload counts:")
print(inference_dataset["workload_type"].value_counts().to_string())
print("\nMissing values:")
print(inference_dataset.isna().sum().to_string())
display(inference_dataset.head())

Dataset shape: (2424, 7)

Workload counts:
workload_type
offline_inference          1200
online_finite_inference    1024
online_rate_inference       200

Missing values:
workload_type           0
batch_size           1224
max_output_tokens    1224
mean_power_w            0
request_rate         1200
output_length        1400
dataset              1200


,workload_type,batch_size,max_output_tokens,mean_power_w,request_rate,output_length,dataset
0,offline_inference,550.0,1024.0,2822.104293,NaN,NaN,NaN
1,offline_inference,550.0,1024.0,2814.350863,NaN,NaN,NaN
2,offline_inference,550.0,1024.0,2798.014121,NaN,NaN,NaN
3,offline_inference,275.0,1024.0,2694.764568,NaN,NaN,NaN
4,offline_inference,275.0,1024.0,2696.992151,NaN,NaN,NaN


## 2.3 Leakage Audit

An explicit leakage audit is performed to ensure that the model uses only workload and configuration information that is appropriate for prediction.

Target derived variables, post execution outcomes, identifiers, and execution metadata are excluded from the predictor set.

The audit also verifies that the target variable, `mean_power_w`, is not included among the model predictors.

In [4]:
feature_columns = ["workload_type", "batch_size", "max_output_tokens",
                   "request_rate", "output_length", "dataset"]
target_column = "mean_power_w"
leakage_fields = {
    "mean_power[W]": "target", "peak_power[W]": "same-execution power outcome",
    "elapsed": "execution outcome", "execution_time_seconds": "measured duration",
    "completed": "execution status", "run_completed": "execution status",
    "id": "identifier", "seed": "identifier", "repeat": "identifier",
    "path_run": "identifier", "slurmid": "identifier",
}
leak_in_features = [c for c in leakage_fields if c in feature_columns]
print("Leakage fields in predictors:", leak_in_features or "NONE")
print("Target in predictors:", target_column in feature_columns)
final_dataset = inference_dataset[feature_columns + [target_column]].copy()
assert final_dataset[target_column].notna().all()
print(f"\nFinal dataset: {final_dataset.shape}, target missing: {final_dataset[target_column].isna().sum()}")

Leakage fields in predictors: NONE
Target in predictors: False

Final dataset: (2424, 7), target missing: 0


## 2.4 Leakage Safe Train / Validation / Test Split

The dataset is split into training, validation, and test sets using a **stratified 70/15/15 split** based on `workload_type`. Stratification ensures that each partition maintains a similar distribution of the three inference workload types.

The split uses `random_state=42` to make the partitioning **reproducible**. Using a fixed random seed ensures that the same rows are assigned to the training, validation, and test sets each time the notebook is run.

The split is performed **before** any preprocessing is fitted to prevent information from the validation or test sets from influencing preprocessing parameters.

Preprocessing parameters will be fitted on the **training partition only** and then applied to the validation and test partitions.

In [5]:
from sklearn.model_selection import train_test_split
X = final_dataset[feature_columns].copy()
y = final_dataset[target_column].copy()
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=X["workload_type"])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=X_temp["workload_type"])
for name, part in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    print(f"{name}: {len(part)} rows | {part['workload_type'].value_counts().to_dict()}")
# No overlap between partitions
assert not (set(X_train.index) & set(X_val.index))
assert not (set(X_train.index) & set(X_test.index))
assert not (set(X_val.index) & set(X_test.index))
print("Partition overlap: 0 rows everywhere")

train: 1696 rows | {'offline_inference': 840, 'online_finite_inference': 716, 'online_rate_inference': 140}
validation: 364 rows | {'offline_inference': 180, 'online_finite_inference': 154, 'online_rate_inference': 30}
test: 364 rows | {'offline_inference': 180, 'online_finite_inference': 154, 'online_rate_inference': 30}
Partition overlap: 0 rows everywhere


## 2.5 Preprocessing Pipeline (Fit on Training Data Only)

- **Numerical features:** Median imputation using training set statistics, missing value indicators, and standard scaling.
- **Categorical features:** Constant `"not_applicable"` imputation followed by one hot encoding with `handle_unknown="ignore"`.
- The preprocessing pipeline is **fitted only on the training partition** and then used to transform the validation and test partitions.
- The fitted preprocessing pipeline is saved to Google Drive so the exact same transformations can be reused during model inference and deployment.

In [7]:
import numpy as np
import joblib
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Define the persistent output directory.
processed_dir = Path(
    "/content/drive/MyDrive/data/processed/modeling"
)
processed_dir.mkdir(
    parents=True,
    exist_ok=True
)
# Define feature groups.
categorical_features = [
    "workload_type",
    "dataset",
]
numerical_features = [
    "batch_size",
    "max_output_tokens",
    "request_rate",
    "output_length",
]
# Build the numerical preprocessing pipeline.
numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
        ),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
])
# Build the categorical preprocessing pipeline.
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="not_applicable",
        ),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])
# Combine numerical and categorical preprocessing.
preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features,
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features,
    ),
])
# Fit the preprocessing pipeline on training data ONLY.
preprocessor.fit(X_train)
# Transform all partitions using the fitted training only pipeline.
X_train_p = preprocessor.transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)
# Retrieve the generated feature names.
feature_names = preprocessor.get_feature_names_out()
# Validate that preprocessing produced finite values.
for name, arr in [
    ("train", X_train_p),
    ("validation", X_val_p),
    ("test", X_test_p),
]:
    assert np.isfinite(arr).all(), (
        f"Non-finite values detected in processed {name} data."
    )
# Save the fitted preprocessing pipeline.
preprocessor_path = processed_dir / "preprocessor.joblib"
joblib.dump(
    preprocessor,
    preprocessor_path,
)
print(
    f"Processed shapes: "
    f"train {X_train_p.shape}, "
    f"validation {X_val_p.shape}, "
    f"test {X_test_p.shape}"
)
print(f"\nNumber of processed features:{len(feature_names)}")
print("\nProcessed features:")
for feature in feature_names:
    print(f"-{feature}")
print(f"\nFitted preprocessor saved to:")
print(preprocessor_path)

Processed shapes: train (1696, 14), validation (364, 14), test (364, 14)

Number of processed features:14

Processed features:
-numerical__batch_size
-numerical__max_output_tokens
-numerical__request_rate
-numerical__output_length
-numerical__missingindicator_batch_size
-numerical__missingindicator_max_output_tokens
-numerical__missingindicator_request_rate
-numerical__missingindicator_output_length
-categorical__workload_type_offline_inference
-categorical__workload_type_online_finite_inference
-categorical__workload_type_online_rate_inference
-categorical__dataset_likaixin/InstructCoder
-categorical__dataset_mgoin/mlperf-inference-llama2-data
-categorical__dataset_not_applicable

Fitted preprocessor saved to:
/content/drive/MyDrive/data/processed/modeling/preprocessor.joblib


## 2.6 Persist Processed Data for `03_Modeling.ipynb`
The processed training, validation, and test datasets are saved to Google Drive so they persist across Colab sessions.

The processed feature matrices and target values are stored as Parquet files for efficient loading in `03_Modeling.ipynb`.

A raw version of the test set is also saved to support per workload evaluation and analysis.

In [8]:
from pathlib import Path
import os
# Save processed datasets to Google Drive for persistence across Colab sessions.
processed_dir = Path(
    "/content/drive/MyDrive/data/processed/modeling"
)
processed_dir.mkdir(parents=True, exist_ok=True)
# Features as DataFrames with proper column names; targets as one column frames.
pd.DataFrame(
    X_train_p, columns=feature_names
).to_parquet(processed_dir / "X_train.parquet")
pd.DataFrame(
    X_val_p, columns=feature_names
).to_parquet(processed_dir / "X_validation.parquet")
pd.DataFrame(
    X_test_p, columns=feature_names
).to_parquet(processed_dir / "X_test.parquet")
y_train.to_frame("mean_power_w").to_parquet(
    processed_dir / "y_train.parquet"
)
y_val.to_frame("mean_power_w").to_parquet(
    processed_dir / "y_validation.parquet"
)
y_test.to_frame("mean_power_w").to_parquet(
    processed_dir / "y_test.parquet"
)
# Raw test features plus target, used for per workload evaluation in 03_Modeling.ipynb.
X_test.assign(
    mean_power_w=y_test
).to_parquet(
    processed_dir / "test_raw.parquet"
)
print(f"Saved to: {processed_dir}")
for file_path in sorted(processed_dir.iterdir()):
    print(f"{file_path.name}")

Saved to: /content/drive/MyDrive/data/processed/modeling
X_test.parquet
X_train.parquet
X_validation.parquet
preprocessor.joblib
test_raw.parquet
y_test.parquet
y_train.parquet
y_validation.parquet
